# Begonia AI - ADIM 3B: Colab T4 QLoRA Eğitimi

Ben bu defteri hocamın Oturum_son_hafta(2).ipynb akışını Begonia AI gerçek verisine uygulamak için hazırladım. Model indirme, eğitim, API çağrısı ve Hub yükleme hücrelerini burada çalıştırmıyorum.

## Akışım

Ben Colab T4 ve paketleri doğrularım; gerçek 450 kayıtlık eğitim verisini hoca şemasına dönüştürürüm; aynı değerlendirme kümesiyle eğitim öncesi ve sonrası ölçerim. Hoca ayarlarını korurum: Qwen2.5-1.5B-Instruct, 4-bit NF4, LoRA r=16, alpha=32, dropout=0.05 ve 3 epoch.

In [ ]:
# Ben bu hücreyi yalnızca Colab T4 oturumunda çalıştırırım.
!pip install -q -U 'transformers>=4.48,<5' 'trl>=0.15,<1' 'peft>=0.14,<1' 'bitsandbytes>=0.45' 'datasets>=3,<4' 'accelerate>=1,<2' 'huggingface_hub>=0.28,<1' pypdf pandas

In [ ]:
import torch
import importlib.metadata as paket_bilgisi

assert torch.cuda.is_available(), 'Ben Colab çalışma zamanını T4 GPU olarak seçmeliyim.'
print('GPU:', torch.cuda.get_device_name(0))
print('Toplam bellek (GB):', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))
for paket in ['torch', 'transformers', 'trl', 'peft', 'bitsandbytes', 'datasets', 'accelerate', 'huggingface_hub', 'pypdf', 'pandas']:
    print(f'{paket}: {paket_bilgisi.version(paket)}')

## Colab dosya düzenim

Ben depoyu /content/begonia-ai altına koyarım veya aynı klasör yapısını Colab'a yüklerim. Referans defterini değiştirmem. ADIM 3A defterini ve ham JSONL dosyalarını korurum; dönüştürülmüş dosyaları ayrı bir klasöre yazarım.

In [ ]:
from pathlib import Path

KOK = Path('/content/begonia-ai') if Path('/content/begonia-ai').exists() else Path.cwd()
KAYNAKLAR = {
    'referans_defter': KOK / 'notebooks' / 'Oturum_son_hafta(2).ipynb',
    'begonia_defteri': KOK / 'notebooks' / 'Begonia_AI_Egitim.ipynb',
    'adim_3a_defteri': KOK / 'notebooks' / 'adim_3a_egitim_hazirligi.ipynb',
    'rehber': KOK / 'data' / 'rehber.pdf', 'odalar': KOK / 'data' / 'odalar.csv',
    'test_sorulari': KOK / 'data' / 'test_sorulari.csv',
    'egitim': KOK / 'data' / 'egitim' / 'egitim.jsonl',
    'dogrulama': KOK / 'data' / 'egitim' / 'dogrulama.jsonl',
    'degerlendirme': KOK / 'data' / 'egitim' / 'degerlendirme.jsonl',
}
EKSIK = [ad for ad, yol in KAYNAKLAR.items() if not yol.exists()]
assert not EKSIK, f'Eksik dosya: {EKSIK}'
print('Kök dizin:', KOK)

## Gerçek verim ve hoca şeması

Ben ham ADIM 3A verisinin izlenebilirlik alanlarını okurum; TRL eğitimine hoca defterindeki gibi yalnızca tur ve messages alanlarını veririm. 150 ajan örneğinin yarısını ajan, yarısını gerçek araç sonucunu takip eden ajan2 olarak temsil ederim. Eğitim kümesi 450 kayıt olarak kalır; eğitim, doğrulama ve değerlendirme ayrımı korunur.

In [ ]:
import inspect
import json
import re
from collections import Counter
from hashlib import sha256

from araclar import hesapla, konaklama_ucreti, oda_bilgisi, rehberde_ara
from proje import ASISTAN_ADI, CEVAP_SONU_ISARETI, KIMLIK_METNI, KURUM_ADI, TABAN_MODEL

ARACLAR = {'rehberde_ara': rehberde_ara, 'oda_bilgisi': oda_bilgisi, 'hesapla': hesapla, 'konaklama_ucreti': konaklama_ucreti}
ARAC_PARAMETRELERI = {ad: tuple(inspect.signature(fonksiyon).parameters) for ad, fonksiyon in ARACLAR.items()}
assert CEVAP_SONU_ISARETI == ''
ARAC_LISTESI = '\n'.join(['- rehberde_ara(sorgu: str): Resmî rehberde arama yapar.', '- oda_bilgisi(oda_sorgusu: str): Odalar CSV tablosunda tam oda türünü arar.', '- hesapla(ifade: str): Güvenli dört işlem yapar.', '- konaklama_ucreti(konaklama_sorgusu: str): Konaklama ücretini hesaplar.'])
AJAN_SISTEM = f'''Sen {ASISTAN_ADI}'sın, {KURUM_ADI} için Türkçe otel resepsiyon asistanısın; soruları araçlarla çözersin.
Araçlar:
{ARAC_LISTESI}
Her adımda YALNIZCA tek JSON nesnesi yaz. Araç için {{"dusunce": "neden", "arac": "araç_adı", "argumanlar": {{"parametre": "değer"}}}}; son cevap için {{"dusunce": "neden", "cevap": "..."}}.
Kurallar: Rehberde, oda tablosunda veya araç sonucunda olmayan bilgi uydurma. Her hesap için hesapla aracını kullan.'''

In [ ]:
def jsonl_oku(yol):
    return [json.loads(satir) for satir in yol.read_text(encoding='utf-8').splitlines() if satir.strip()]

def _soru(kayit):
    return next(mesaj['content'] for mesaj in kayit['messages'] if mesaj['role'] == 'user')

def hoca_kaydina_donustur(kayit, ajan2=False):
    soru, cevap = _soru(kayit), kayit['messages'][-1]['content']
    if kayit['tur'] == 'kimlik':
        return {'tur': 'kimlik', 'messages': [{'role': 'user', 'content': soru}, {'role': 'assistant', 'content': cevap}]}
    if kayit['tur'] == 'rag':
        sistem = f'{KIMLIK_METNI}\nAşağıdaki bağlamı kullanarak cevap ver. Cevap bağlamda yoksa uydurma; resmî resepsiyona yönlendir.\n\nBağlam:\n{kayit["baglam"]}'
        return {'tur': 'rag', 'messages': [{'role': 'system', 'content': sistem}, {'role': 'user', 'content': soru}, {'role': 'assistant', 'content': cevap}]}
    assert kayit['tur'] == 'ajan' and len(kayit['arac_cagrilari']) == 1
    cagri = kayit['arac_cagrilari'][0]
    karar = {'dusunce': f"Bu soruyu {cagri['ad']} aracıyla doğrulamalıyım.", 'arac': cagri['ad'], 'argumanlar': cagri['argumanlar']}
    mesajlar = [{'role': 'system', 'content': AJAN_SISTEM}, {'role': 'user', 'content': soru}, {'role': 'assistant', 'content': json.dumps(karar, ensure_ascii=False)}]
    if not ajan2:
        return {'tur': 'ajan', 'messages': mesajlar}
    son_karar = {'dusunce': 'Araç sonucu doğrulandı; kısa ve kaynakla sınırlı cevap verebilirim.', 'cevap': cevap}
    return {'tur': 'ajan2', 'messages': mesajlar + [{'role': 'user', 'content': f"Araç sonucu: {cagri['sonuc']}"}, {'role': 'assistant', 'content': json.dumps(son_karar, ensure_ascii=False)}]}

def hoca_kumesine_donustur(kayitlar):
    sonuc, ajan_sira = [], 0
    for kayit in kayitlar:
        sonuc.append(hoca_kaydina_donustur(kayit, ajan2=(kayit['tur'] == 'ajan' and ajan_sira % 2 == 1)))
        ajan_sira += kayit['tur'] == 'ajan'
    return sonuc

In [ ]:
HAM_KUMELER = {ad: jsonl_oku(KAYNAKLAR[ad]) for ad in ('egitim', 'dogrulama', 'degerlendirme')}
HOCA_KUMELERI = {ad: hoca_kumesine_donustur(kayitlar) for ad, kayitlar in HAM_KUMELER.items()}

def soru_imzasi(kayit):
    return sha256(re.sub(r'\W+', '', _soru(kayit).casefold()).encode('utf-8')).hexdigest()

assert len(HOCA_KUMELERI['egitim']) == 450
assert {'kimlik', 'rag', 'ajan', 'ajan2'} <= {kayit['tur'] for kayit in HOCA_KUMELERI['egitim']}
for ilk, ikinci in [('egitim', 'dogrulama'), ('egitim', 'degerlendirme'), ('dogrulama', 'degerlendirme')]:
    assert not ({soru_imzasi(k) for k in HOCA_KUMELERI[ilk]} & {soru_imzasi(k) for k in HOCA_KUMELERI[ikinci]})
for ad, kume in HOCA_KUMELERI.items():
    print(ad, len(kume), dict(Counter(k['tur'] for k in kume)))

COLAB_VERI = KOK / 'data' / 'egitim' / 'hoca_uyumlu'
COLAB_VERI.mkdir(parents=True, exist_ok=True)
for ad, kayitlar in HOCA_KUMELERI.items():
    (COLAB_VERI / f'{ad}.jsonl').write_text('\n'.join(json.dumps(k, ensure_ascii=False) for k in kayitlar) + '\n', encoding='utf-8')

## Eğitim öncesi ölçümüm

Ben eğitim öncesinde ve sonrasında aynı 60 kayıtlık değerlendirme kümesini kullanırım. Kimlik, RAG, geçerli araç JSON'u ve doğru araç/argüman birlikte ölçülür. RAG puanını dış hakem API'si yerine kaynak kaydındaki zorunlu ifadelerle yerel ve tekrarlanabilir biçimde hesaplarım.

In [ ]:
EGITIM_CALISTIRMA_IZNI = False

QLORA_AYARLARI = {'r': 16, 'lora_alpha': 32, 'lora_dropout': 0.05, 'bias': 'none', 'task_type': 'CAUSAL_LM', 'target_modules': ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']}
SFT_AYARLARI = {'num_train_epochs': 3, 'per_device_train_batch_size': 4, 'gradient_accumulation_steps': 4, 'learning_rate': 2e-4, 'lr_scheduler_type': 'cosine', 'warmup_steps': 5, 'logging_steps': 10, 'save_strategy': 'epoch', 'save_total_limit': 2, 'eval_strategy': 'epoch', 'per_device_eval_batch_size': 4, 'fp16': True, 'bf16': False, 'max_length': 1024, 'gradient_checkpointing': True, 'optim': 'paged_adamw_8bit', 'report_to': 'none'}
assert QLORA_AYARLARI['r'] == 16 and QLORA_AYARLARI['lora_alpha'] == 32 and QLORA_AYARLARI['lora_dropout'] == 0.05
assert SFT_AYARLARI['num_train_epochs'] == 3

In [ ]:
# Ben bu hücreyi yalnızca Colab'da eğitim iznini True yaptıktan sonra çalıştırırım.
assert EGITIM_CALISTIRMA_IZNI, 'Model indirme ve eğitim için önce EGITIM_CALISTIRMA_IZNI değerini True yapmalıyım.'
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
tok = AutoTokenizer.from_pretrained(TABAN_MODEL)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(TABAN_MODEL, quantization_config=bnb, device_map='auto', torch_dtype=torch.float16)
model.config.use_cache = False

In [ ]:
def uret(mesajlar, m, maks=160):
    girdi = tok.apply_chat_template(mesajlar, add_generation_prompt=True, return_tensors='pt', return_dict=True).to(m.device)
    with torch.no_grad():
        cikti = m.generate(**girdi, max_new_tokens=maks, do_sample=False, pad_token_id=tok.eos_token_id)
    return tok.decode(cikti[0, girdi['input_ids'].shape[1]:], skip_special_tokens=True).strip()

def degerlendir_model(m, etiket):
    sayac, payda = Counter(), Counter()
    for ham, hoca in zip(HAM_KUMELER['degerlendirme'], HOCA_KUMELERI['degerlendirme']):
        if ham['tur'] == 'ajan':
            yanit = uret(hoca['messages'][:2], m, 120)
            payda['ajan_json'] += 1; payda['ajan_arac'] += 1
            try:
                karar = json.loads(yanit[yanit.index('{'):yanit.rindex('}') + 1])
                sayac['ajan_json'] += 1
                beklenen = json.loads(hoca['messages'][2]['content'])
                sayac['ajan_arac'] += int(karar.get('arac') == beklenen['arac'] and karar.get('argumanlar') == beklenen['argumanlar'])
            except (ValueError, json.JSONDecodeError):
                pass
        else:
            yanit = uret(hoca['messages'][:-1], m, 120)
            if ham['tur'] == 'kimlik':
                payda['kimlik'] += 1; sayac['kimlik'] += int(ASISTAN_ADI.casefold() in yanit.casefold() and KURUM_ADI.casefold() in yanit.casefold())
            elif ham['tur'] == 'rag':
                payda['rag'] += 1; sayac['rag'] += int(all(i.casefold() in yanit.casefold() for i in ham['beklenti']['rag_zorunlu_ifadeler']))
    puan = {ad: round(sayac[ad] / payda[ad], 4) for ad in payda}
    print(etiket, puan)
    return puan

# Ben bu ölçümü taban model yüklendikten sonra, eğitimden hemen önce çalıştırırım.
ONCE = degerlendir_model(model, 'ÖNCE: ham Qwen2.5-1.5B-Instruct')

## QLoRA eğitimim ve checkpoint'lerim

Ben TRL veri kümesine yalnızca hoca şemasındaki messages alanını veririm. Her epoch sonunda checkpoint kaydeder, en son iki checkpoint'i saklarım. Ham veri ve değerlendirme verisi aynı kalır.

In [ ]:
from datasets import load_dataset
from peft import LoraConfig, prepare_model_for_kbit_training
from trl import SFTConfig, SFTTrainer

veri = load_dataset('json', data_files={'train': str(COLAB_VERI / 'egitim.jsonl'), 'test': str(COLAB_VERI / 'dogrulama.jsonl')})
veri = veri.remove_columns(['tur'])
model = prepare_model_for_kbit_training(model)
lora = LoraConfig(**QLORA_AYARLARI)
ayar = SFTConfig(output_dir=str(KOK / 'begonia-ai-lora'), **SFT_AYARLARI)
egitici = SFTTrainer(model=model, args=ayar, peft_config=lora, train_dataset=veri['train'], eval_dataset=veri['test'], processing_class=tok)
for parametre in egitici.model.parameters():
    if parametre.requires_grad:
        parametre.data = parametre.data.float()
egitici.model.print_trainable_parameters()

In [ ]:
# Ben eğitimi yalnızca eğitim izni açıkken başlatırım.
assert EGITIM_CALISTIRMA_IZNI
sonuc = egitici.train()
print(f'Ben eğitimi tamamladım; ortalama loss: {sonuc.training_loss:.4f}')
for kayit in egitici.state.log_history:
    if 'eval_loss' in kayit:
        print(f"epoch {kayit['epoch']:.0f}: doğrulama loss {kayit['eval_loss']:.4f}")
egitici.save_model(str(KOK / 'begonia-ai-lora' / 'son'))
tok.save_pretrained(str(KOK / 'begonia-ai-lora' / 'son'))

## Eğitim sonrası ölçümüm ve karşılaştırma raporum

Ben eğitim öncesindeki aynı 60 değerlendirme kaydı ve aynı üretim ayarlarıyla tekrar ölçerim. Tablo, ölçüt bazında önce, sonra ve fark değerlerini gösterir.

In [ ]:
begonia = egitici.model
begonia.eval()
SONRA = degerlendir_model(begonia, 'SONRA: Begonia AI LoRA')

import pandas as pd
rapor = pd.DataFrame({'önce': ONCE, 'sonra': SONRA})
rapor['fark'] = rapor['sonra'] - rapor['önce']
display(rapor.style.format('{:.2%}'))

## Hugging Face yüklemem

Ben token'ı yalnızca Colab Secrets içindeki HF_TOKEN anahtarından okurum. Token değerini yazdırmam, dosyaya kaydetmem ve notebook çıktısına koymam. Bu bölüm ayrı izin gerektirir ve varsayılan olarak kapalıdır.

In [ ]:
HF_YUKLEME_IZNI = False
assert HF_YUKLEME_IZNI, 'Hub yüklemesi için önce HF_YUKLEME_IZNI değerini True yapmalıyım.'
from google.colab import userdata
from huggingface_hub import HfApi, login, whoami

hf_token = userdata.get('HF_TOKEN')
assert hf_token, 'Ben Colab Secrets içine HF_TOKEN eklemeliyim.'
login(token=hf_token, add_to_git_credential=False)
KULLANICI = whoami()['name']

In [ ]:
# Ben bu hücreyi yalnızca eğitim ve ölçüm sonuçlarını gözden geçirdikten sonra çalıştırırım.
ADAPTOR = f'{KULLANICI}/Begonia-AI-1.5B-LoRA'
TAM_MODEL = f'{KULLANICI}/Begonia-AI-1.5B'
VERI_SETI = f'{KULLANICI}/begonia-ai-egitim-verisi'
egitici.model.push_to_hub(ADAPTOR, private=False)
tok.push_to_hub(ADAPTOR)
api = HfApi()
api.create_repo(VERI_SETI, repo_type='dataset', exist_ok=True, private=False)
for ad in ('egitim', 'dogrulama', 'degerlendirme'):
    api.upload_file(path_or_fileobj=str(COLAB_VERI / f'{ad}.jsonl'), path_in_repo=f'{ad}.jsonl', repo_id=VERI_SETI, repo_type='dataset')

In [ ]:
# Ben tam modeli yalnızca adaptör ve veri seti yüklemesi onaylandıktan sonra birleştiririm.
import gc
from peft import PeftModel

del egitici, model, begonia
gc.collect(); torch.cuda.empty_cache()
taban16 = AutoModelForCausalLM.from_pretrained(TABAN_MODEL, torch_dtype=torch.float16, device_map='auto')
begonia = PeftModel.from_pretrained(taban16, str(KOK / 'begonia-ai-lora' / 'son')).merge_and_unload()
begonia.save_pretrained(str(KOK / 'Begonia-AI-1.5B'), safe_serialization=True)
tok.save_pretrained(str(KOK / 'Begonia-AI-1.5B'))
begonia.push_to_hub(TAM_MODEL, private=False)
tok.push_to_hub(TAM_MODEL)

## Çalıştırma sınırım

Bu defteri hazırlarken model indirmedim, GPU eğitimi başlatmadım, dış API çağırmadım ve Hugging Face'e yükleme yapmadım. Colab'da önce veri ve ölçüm çıktılarını gözden geçiririm; eğitim için EGITIM_CALISTIRMA_IZNI, Hub yüklemesi için HF_YUKLEME_IZNI değerini ancak bilinçli onayla açarım.